# Thermofluids Flow Network

In order to model our data center, we can build a simplified flow network by treating the fluid system as an electrical circuit analog. This will help us in understanding the acausal equation-solving mechanics that Modelica does. 

We can represent the data center cooling loops (pumps, chillers, server racks) as a graph of nodes and branches. We can then write a simple system of non-linear equations to solve for unknown pressures and mass flow. 

* **Node $i$ Control Volume Equations**

Nodes represent lumped control volumes with mass $m_i$, volume $V_i$, and total energy $E_i = U_i + \frac{1}{2} m_i v_i^2 + m_i g z_i$. Flow enters node $i$ from neighbor nodes $k \in \text{in}(i)$ and leaves node $i$ toward neighbor nodes $l \in \text{out}(i)$.

* **Conservation of Mass:**

$$\frac{d m_i}{dt} = \sum_{k \in \text{in}(i)} \dot{m}_{ki} - \sum_{l \in \text{out}(i)} \dot{m}_{il}$$


* **Conservation of Energy:**

$$\frac{d E_i}{dt} = \dot{Q}_i - \dot{W}_i + \sum_{k \in \text{in}(i)} \dot{m}_{ki} \left( h_k + \frac{v_k^2}{2} + g z_k \right) - \sum_{l \in \text{out}(i)} \dot{m}_{il} \left( h_i + \frac{v_i^2}{2} + g z_i \right)$$


* $\dot{Q}_i$: External heat addition rate across node $i$ boundary (e.g., direct thermal dissipation)
* $\dot{W}_i$: Boundary or shaft work rate done by control volume node $i$

**Branch $ij$ Control Volume Equations**

Branch $ij$ connects upstream node $i$ to downstream node $j$ with length $L_{ij}$ and cross-sectional area $A_{ij}$.

* **Conservation of Mass:**

$$\frac{d m_{ij}}{dt} = \dot{m}_{ij, \text{in}} - \dot{m}_{ij, \text{out}}$$


* **Conservation of Momentum (1D Integrated Navier-Stokes):**

Integrating the 3D Cauchy momentum equation along a 1D stream-oriented coordinate $s$ directly yields the dynamic branch pressure drop.

**1. Generalized 3D Momentum Equation**

The differential conservation of linear momentum for an incompressible, viscous fluid in a gravitational field is given by:

$$\rho \left( \frac{\partial \mathbf{v}}{\partial t} + \mathbf{v} \cdot \nabla \mathbf{v} \right) = -\nabla P + \nabla \cdot \boldsymbol{\tau} + \rho \mathbf{g} + \mathbf{f}_{\text{ext}}$$

where $\mathbf{v}$ is the 3D velocity vector, $P$ is static pressure, $\boldsymbol{\tau}$ is the viscous stress tensor, $\mathbf{g}$ is gravitational acceleration, and $\mathbf{f}_{\text{ext}}$ represents external body forces per unit volume (e.g., pump momentum insertion).

**2. 1D Reduction along the Streamline**

Assume the flow is predominantly 1-dimensional along path $s \in [0, L_{ij}]$ with unit tangent vector $\hat{e}_s$, reducing velocity to scalar $v(s, t) = \mathbf{v} \cdot \hat{e}_s$. Projecting the 3D momentum equation along $s$:

$$\rho \left( \frac{\partial v}{\partial t} + v \frac{\partial v}{\partial s} \right) = -\frac{\partial P}{\partial s} - \rho g \frac{dz}{ds} - f_{\text{viscous}} + f_{\text{source}}$$

where:

* $z(s)$ is the vertical elevation coordinate along the branch length.
* $f_{\text{viscous}} = -(\nabla \cdot \boldsymbol{\tau}) \cdot \hat{e}_s$ is the internal viscous resistance force per unit volume.
* $f_{\text{source}} = \mathbf{f}_{\text{ext}} \cdot \hat{e}_s$ is the active pump pressure gradient per unit volume.

**3. Spatial Integration across Branch $ij$**

Integrate the differential equation along the path length from node $i$ ($s=0$) to node $j$ ($s=L_{ij}$):

$$\int_{0}^{L_{ij}} \rho \frac{\partial v}{\partial t} \, ds + \int_{0}^{L_{ij}} \rho v \frac{\partial v}{\partial s} \, ds = -\int_{P_i}^{P_j} dP - \int_{z_i}^{z_j} \rho g \, dz - \int_{0}^{L_{ij}} f_{\text{viscous}} \, ds + \int_{0}^{L_{ij}} f_{\text{source}} \, ds$$

Evaluating each integral individually:

* **Transient Inertia:**
Expressing local velocity via mass flow rate $v(s, t) = \frac{\dot{m}_{ij}(t)}{\rho A(s)}$:

$$\int_{0}^{L_{ij}} \rho \frac{\partial}{\partial t} \left( \frac{\dot{m}_{ij}}{\rho A(s)} \right) ds = \left( \int_{0}^{L_{ij}} \frac{ds}{A(s)} \right) \frac{d\dot{m}_{ij}}{dt}$$

For a uniform cross-sectional area $A_{ij}$, this simplifies to $\frac{L_{ij}}{A_{ij}} \frac{d\dot{m}_{ij}}{dt}$.
* **Advective Kinetic Energy Change:**

$$\int_{0}^{L_{ij}} \rho v \frac{\partial v}{\partial s} \, ds = \int_{v_i}^{v_j} \rho v \, dv = \frac{1}{2}\rho \left( v_j^2 - v_i^2 \right)$$


* **Static Pressure Drop:**

$$-\int_{P_i}^{P_j} dP = P_i - P_j$$


* **Gravitational Potential Change:**

$$-\int_{z_i}^{z_j} \rho g \, dz = -\rho g (z_j - z_i) = \rho g (z_i - z_j)$$


* **Viscous Friction Losses:**

$$\int_{0}^{L_{ij}} f_{\text{viscous}} \, ds = \Delta P_{\text{loss}, ij} = \frac{1}{2}\rho v^2 \left( \frac{f_D L_{ij}}{D_{h, ij}} + \sum K_{ij} \right)$$

**$D_{h, ij}$ — Hydraulic Diameter ($\text{m}$)**

The hydraulic diameter generalizes non-circular conduit cross-sections so standard pipe friction correlations (like the Darcy-Weisbach friction factor $f_D$) can be applied:

$$D_{h, ij} = \frac{4 A_{ij}}{P_{w, ij}}$$

* $A_{ij}$: Cross-sectional flow area ($\text{m}^2$)
* $P_{w, ij}$: Wetted perimeter ($\text{m}$), which is the perimeter of the cross-section in physical contact with the flowing fluid.

*Examples in Data Centers:* For a circular pipe, $D_h$ reduces to the internal diameter $D$. For rectangular containment aisles, server chassis airflow channels, or cold-plate microchannels, $D_h$ accounts for the increased wall shear relative to the open area.

**$\sum K_{ij}$ — Sum of Minor Loss Coefficients (Dimensionless)**

This term accounts for localized dynamic pressure drops caused by component geometries, fittings, and flow obstructions along branch $ij$:

$$\sum K_{ij} = K_{\text{bends}} + K_{\text{valves}} + K_{\text{couplings}} + K_{\text{expansions/contractions}} + K_{\text{coils}}$$

* **Distributed vs. Local Losses:** While the term $\frac{f_D L_{ij}}{D_{h, ij}}$ measures continuous skin friction shear along a straight length of pipe or duct, $K$ values capture localized turbulence, flow separation, and secondary flows.
* **Data Center Context:** In liquid or air loops, minor losses often dominate major pipe losses. $K$ factors represent pressure drops across server intake grilles, quick-disconnect fittings on direct-to-chip loops, 90° pipe elbows, CRAH/CRAC cooling coils, or flow control valves. Each factor scales the local dynamic pressure ($\frac{1}{2}\rho v^2$) into an irreversible static pressure drop.

* **Active Mechanical Work Source:**

$$\int_{0}^{L_{ij}} f_{\text{source}} \, ds = \Delta P_{\text{source}, ij}$$


**4. Final Algebraic System Equation**

Combining terms and isolating the node potential difference $(P_i - P_j)$:

$$P_i - P_j = \underbrace{\left( \int_{0}^{L_{ij}} \frac{ds}{A(s)} \right) \frac{d\dot{m}_{ij}}{dt}}_{\text{Unsteady Inertia}} + \underbrace{\frac{1}{2}\rho \left( v_j^2 - v_i^2 \right)}_{\text{Kinetic Energy Diff.}} + \underbrace{\rho g (z_j - z_i)}_{\text{Potential Energy Diff.}} + \underbrace{\Delta P_{\text{loss}, ij}}_{\text{Viscous Losses}} - \underbrace{\Delta P_{\text{source}, ij}}_{\text{Pump Head Input}}$$

* **Conservation of Energy:**

$$\frac{d E_{ij}}{dt} = \dot{Q}_{ij} - \dot{W}_{ij} + \dot{m}_{ij, \text{in}} \left( h_i + \frac{v_i^2}{2} + g z_i \right) - \dot{m}_{ij, \text{out}} \left( h_j + \frac{v_j^2}{2} + g z_j \right)$$